# Problema 5 — Los ODS que Nadie Monitorea
## ODS 2 (Hambre Cero) × ODS 14 (Vida Submarina) × El vacío estadístico

> No podemos gestionar lo que no medimos. México tiene 333 indicadores ODS — pero
> para muchos de ellos el casillero está en blanco. Lo que no se registra no se combate,
> y lo que no se combate empeora en silencio.

---
| Acto | Ángulo |
|------|--------|
| I | El mapa del silencio — cobertura por ODS |
| II | ODS 2: el hambre en cámara lenta |
| III | ODS 14: el océano con 4 indicadores |
| IV | La salud de los datos — distribución del vacío |
| V | 18 años sin moverse — malnutrición vs el mundo |
| VI | El presupuesto hundido y los indicadores fantasma |
| VII | Radar del silencio — los 17 ODS evaluados |

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Datos ─────────────────────────────────────────────────────────────────────
df   = pd.read_csv('../data/consolidated/indicadores_largo.csv')
meta = pd.read_csv('../data/consolidated/metadatos_indicadores.csv')

def get_serie(ind_id):
    return df[df['indicator_id'] == ind_id].sort_values('periodo').copy()

# ── Estadísticas de cobertura por ODS ─────────────────────────────────────────
cov = meta.groupby('ods_number').agg(
    n_total   =('indicator_id', 'count'),
    n_datos   =('n_observations', lambda x: (x > 0).sum()),
    n_ricos   =('n_observations', lambda x: (x >= 15).sum()),
    n_zero    =('n_observations', lambda x: (x == 0).sum()),
    n_escasos =('n_observations', lambda x: ((x > 0) & (x < 5)).sum()),
    n_medios  =('n_observations', lambda x: ((x >= 5) & (x < 15)).sum()),
    avg_obs   =('n_observations', 'mean')
).reset_index()
cov['pct_datos'] = (cov['n_datos'] / cov['n_total'] * 100).round(1)
cov['pct_ricos'] = (cov['n_ricos'] / cov['n_total'] * 100).round(1)

# ── Paleta de colores ─────────────────────────────────────────────────────────
C_ODS2   = '#DDA63A'   # amarillo ODS 2
C_ODS14  = '#0A97D9'   # azul ODS 14
C_DATO   = '#2DC653'   # verde — tenemos datos
C_VACIO  = '#E63946'   # rojo — sin datos
C_ESCASO = '#F4A261'   # naranja — datos escasos
C_MEDIO  = '#FFD166'   # amarillo — datos medios
C_FONDO  = '#F8F9FA'
C_LINEA  = '#343A40'

print('Setup completo. {} indicadores en {} ODS.'.format(len(meta), meta['ods_number'].nunique()))
print('Indicadores sin ningun dato: {}'.format((meta['n_observations']==0).sum()))
print('ODS con peor cobertura:')
print(cov.sort_values('pct_datos')[['ods_number','n_total','n_datos','pct_datos','avg_obs']].head(5).to_string())

Setup completo. 333 indicadores en 17 ODS.
Indicadores sin ningun dato: 41
ODS con peor cobertura:
   ods_number  n_total  n_datos  pct_datos    avg_obs
0           1       31       18       58.1   5.903226
1           2       17       12       70.6   4.941176
3           4       33       27       81.8   8.424242
9          10       11        9       81.8   4.727273
2           3       42       37       88.1  18.214286


## Acto I — El Mapa del Silencio
De los 17 ODS, sólo 8 tienen el **100 %** de sus indicadores con algún dato.
El **ODS 1** (Fin de la Pobreza) es el peor: apenas el **58 %** de sus indicadores reporta datos —
incluyendo vacíos críticos en protección social y pobreza multidimensional a nivel municipal.
El **ODS 2** (Hambre Cero) tiene cobertura del **70.6 %** y en promedio sólo **4.9 observaciones
por indicador** — menos de una medición cada cuatro años.

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 1 — El Mapa del Silencio
# ══════════════════════════════════════════════════════════════════════════════
ods_labels = [
    '1 Pobreza', '2 Hambre', '3 Salud', '4 Educacion',
    '5 Genero', '6 Agua', '7 Energia', '8 Trabajo',
    '9 Industria', '10 Desigualdad', '11 Ciudades', '12 Consumo',
    '13 Clima', '14 Oceanos', '15 Tierra', '16 Paz', '17 Alianzas'
]

color_cob = [
    C_VACIO if v < 70 else C_ESCASO if v < 85 else C_DATO if v < 100 else '#1A7F3C'
    for v in cov['pct_datos']
]
color_ric = [
    C_VACIO if v < 10 else C_ESCASO if v < 30 else C_DATO if v < 60 else '#1A7F3C'
    for v in cov['pct_ricos']
]

fig1 = go.Figure()

fig1.add_trace(go.Bar(
    name='% indicadores con datos (cobertura)',
    x=ods_labels,
    y=cov['pct_datos'],
    marker_color=color_cob,
    marker_line_color='white', marker_line_width=0.5,
    offsetgroup=0,
    text=['{:.0f}%'.format(v) for v in cov['pct_datos']],
    textposition='outside', textfont=dict(size=8),
    hovertemplate='<b>%{x}</b><br>Cobertura: <b>%{y:.1f}%</b><extra></extra>'
))

fig1.add_trace(go.Bar(
    name='% indicadores con >=15 datos (profundidad)',
    x=ods_labels,
    y=cov['pct_ricos'],
    marker_color=color_ric,
    marker_line_color='white', marker_line_width=0.5,
    opacity=0.72,
    offsetgroup=1,
    text=['{:.0f}%'.format(v) for v in cov['pct_ricos']],
    textposition='outside', textfont=dict(size=8),
    hovertemplate='<b>%{x}</b><br>Con >=15 obs: <b>%{y:.1f}%</b><extra></extra>'
))

# Anotaciones críticas
fig1.add_annotation(
    x='1 Pobreza', y=64,
    text='<b>Peor:</b><br>58.1% cobertura<br>13 indicadores<br>sin ningun dato',
    showarrow=True, arrowhead=2, ax=0, ay=55,
    font=dict(size=9, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_VACIO, borderwidth=1.5
)
fig1.add_annotation(
    x='2 Hambre', y=76,
    text='<b>ODS 2:</b><br>70.6% cobertura<br>prom. 4.9 obs/<br>indicador',
    showarrow=True, arrowhead=2, ax=0, ay=45,
    font=dict(size=9, color=C_ESCASO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ESCASO, borderwidth=1.5
)
fig1.add_annotation(
    x='5 Genero', y=99,
    text='<b>ODS 5:</b> 93% cobertura<br>pero 0% con<br>>=15 datos',
    showarrow=True, arrowhead=2, ax=45, ay=-30,
    font=dict(size=9, color=C_ESCASO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ESCASO, borderwidth=1.5
)
fig1.add_annotation(
    x='14 Oceanos', y=106,
    text='<b>ODS 14:</b> 100% cobertura<br>PERO solo 4 indicadores<br>para todo el oceano',
    showarrow=True, arrowhead=2, ax=0, ay=-45,
    font=dict(size=9, color=C_ODS14),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ODS14, borderwidth=1.5
)

fig1.add_hline(
    y=100, line_width=1, line_dash='dash', line_color='#ADB5BD',
    annotation_text='Cobertura total (100%)',
    annotation_position='right',
    annotation_font=dict(size=9, color='#6C757D')
)

fig1.update_layout(
    title=dict(
        text='<b>El Mapa del Silencio — Cobertura y Profundidad de Datos por ODS</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Barras oscuras = % con algun dato; barras claras = % con 15+ observaciones (analizables)</span>',
        x=0.5, xanchor='center'
    ),
    barmode='group',
    height=520, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    xaxis=dict(tickangle=-35, showgrid=False),
    yaxis=dict(range=[0, 120], showgrid=True, gridcolor='#E9ECEF', title_text='% de indicadores'),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.30, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=110, b=110)
)
fig1.show()

## Acto II — ODS 2: El Hambre en Cámara Lenta
En **18 años** el porcentaje de niños menores de 5 años con retraso en crecimiento (stunting)
pasó de **15.4 % (2006) a 15.1 % (2024)**: esencialmente sin movimiento.
La inseguridad alimentaria moderada o severa afectó al **20 % de los mexicanos** en 2020
y apenas se redujo a **16.2 %** en 2022. El rendimiento agrícola del maíz sí mejora
— pero parte tan bajo que en 2022 sigue **40 % por debajo del promedio mundial**.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 2 — ODS 2: El Hambre en Cámara Lenta
# ══════════════════════════════════════════════════════════════════════════════
stunting    = get_serie('2.2.1')       # retraso en crecimiento
malnutricion = get_serie('2N.1.1')     # malnutricion <5 años
inseq_hist  = get_serie('2.1.2(2)')    # inseg alimentaria 2008-2014
inseq_rec   = get_serie('2.1.2(1)')    # inseg alimentaria 2016-2022
rendimiento = get_serie('2N.3.1')      # rendimiento maíz ton/ha

# Unir series de inseguridad alimentaria
inseq = pd.concat([inseq_hist, inseq_rec]).sort_values('periodo').drop_duplicates('periodo')

fig2 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Inseguridad Alimentaria Moderada/Severa (%)',
        'Retraso en Crecimiento — Stunting (<5 anos, %)',
        'Rendimiento Agricola del Maiz (ton/ha)'
    ],
    horizontal_spacing=0.10
)

# ── Panel 1: Inseguridad alimentaria ─────────────────────────────────────────
inseq_colors = [C_VACIO if v > 22 else C_ESCASO if v > 18 else C_ODS2 for v in inseq['valor']]
fig2.add_trace(go.Bar(
    x=inseq['periodo'], y=inseq['valor'],
    marker_color=inseq_colors,
    marker_line_color='white', marker_line_width=0.5,
    text=['{:.1f}%'.format(v) for v in inseq['valor']],
    textposition='outside', textfont=dict(size=9),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Inseg. alimentaria: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig2.add_annotation(
    x=2010, y=26,
    text='<b>24.8%</b><br>Pico 2010',
    showarrow=True, arrowhead=2, ax=0, ay=-30,
    font=dict(size=9, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_VACIO, borderwidth=1,
    row=1, col=1
)
fig2.add_annotation(
    x=2022, y=18,
    text='<b>16.2%</b><br>2022: mejora<br>pero 1 de cada 6',
    showarrow=True, arrowhead=2, ax=0, ay=-45,
    font=dict(size=9, color=C_ODS2),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ODS2, borderwidth=1.5,
    row=1, col=1
)
# Span de datos faltantes 2015
fig2.add_vrect(
    x0=2014.5, x1=2015.5,
    fillcolor='rgba(173,181,189,0.2)', line_width=0,
    annotation_text='sin dato', annotation_position='top',
    annotation_font=dict(size=8, color='#6C757D'),
    row=1, col=1
)

# ── Panel 2: Stunting ─────────────────────────────────────────────────────────
st_colors = [C_VACIO if v > 14 else C_ESCASO if v > 10 else C_DATO for v in stunting['valor']]
fig2.add_trace(go.Scatter(
    x=stunting['periodo'], y=stunting['valor'],
    mode='lines+markers',
    line=dict(color=C_ODS2, width=3),
    marker=dict(size=11, color=st_colors, line=dict(width=2, color='white')),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Stunting: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

fig2.add_hline(
    y=5, line_width=1.5, line_dash='dash', line_color=C_DATO,
    annotation_text='Meta ODS 2030: <5%',
    annotation_position='right',
    annotation_font=dict(size=9, color=C_DATO),
    row=1, col=2
)
fig2.add_annotation(
    x=2006, y=15.4,
    text='<b>15.4%</b><br>2006',
    showarrow=False, font=dict(size=9, color=C_VACIO), row=1, col=2
)
fig2.add_annotation(
    x=2024, y=15.06,
    text='<b>15.1%</b><br>2024<br>-0.3 pp<br>en 18 anos',
    showarrow=True, arrowhead=2, ax=50, ay=0,
    font=dict(size=9, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_VACIO, borderwidth=1.5,
    row=1, col=2
)

# ── Panel 3: Rendimiento maíz ─────────────────────────────────────────────────
fig2.add_trace(go.Scatter(
    x=rendimiento['periodo'], y=rendimiento['valor'],
    mode='lines',
    line=dict(color=C_ODS2, width=3),
    fill='tozeroy', fillcolor='rgba(221,166,58,0.10)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Rendimiento maiz: <b>%{y:.2f} ton/ha</b><extra></extra>'
), row=1, col=3)

# Referencia mundial
fig2.add_hline(
    y=5.8, line_width=1.5, line_dash='dash', line_color='#6C757D',
    annotation_text='Promedio mundial ~5.8 ton/ha',
    annotation_position='right',
    annotation_font=dict(size=9, color='#6C757D'),
    row=1, col=3
)
fig2.add_annotation(
    x=2000, y=2.46, text='2000:<br>2.5 ton/ha', showarrow=False,
    font=dict(size=9, color=C_VACIO), row=1, col=3
)
fig2.add_annotation(
    x=2022, y=3.9,
    text='<b>2022: 3.9</b><br>+59% desde 2000<br>pero 33% abajo<br>del promedio mundial',
    showarrow=True, arrowhead=2, ax=-80, ay=-35,
    font=dict(size=9, color=C_ESCASO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ESCASO, borderwidth=1.5,
    row=1, col=3
)

fig2.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig2.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig2.update_yaxes(title_text='% poblacion', row=1, col=1)
fig2.update_yaxes(title_text='% menores de 5 anos', row=1, col=2)
fig2.update_yaxes(title_text='toneladas / hectarea', row=1, col=3)

fig2.update_layout(
    title=dict(
        text='<b>ODS 2 — El Hambre en Camara Lenta</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Stunting esencialmente plano en 18 anos; inseguridad alimentaria en 1 de cada 6 mexicanos</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig2.show()

## Acto III — ODS 14: El Océano con 4 Indicadores
México tiene **11,592 km de litoral** — el 11º más largo del mundo — y para todo su ecosistema
marino registra **exactamente 4 indicadores ODS**. El logro es real: las áreas marinas protegidas
saltaron de 1.3 % a **22.5 %** gracias al decreto de Revillagigedo en 2017. Los manglares
se recuperaron (+18 % desde 2010). Pero el presupuesto federal de investigación marina
**se desplomó 78 %** desde su pico de 2012 — y decenas de indicadores críticos como
la acidificación del océano, el estado de los bancos pesqueros y la contaminación costera
simplemente **no se miden**.

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 3 — ODS 14: El Océano con 4 Indicadores
# ══════════════════════════════════════════════════════════════════════════════
areas_marinas = get_serie('14.5.1')
manglares     = get_serie('14R.2')
presupuesto   = get_serie('14.A.1')

fig3 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Areas Marinas Protegidas (% zona maritima)',
        'Superficie de Manglares (miles de ha)',
        'Presupuesto Investigacion Marina (% fed. I+D)'
    ],
    horizontal_spacing=0.11
)

# ── Panel 1: Áreas marinas protegidas ────────────────────────────────────────
am_colors = [C_DATO if v > 10 else C_ESCASO if v > 1 else C_VACIO for v in areas_marinas['valor']]
fig3.add_trace(go.Scatter(
    x=areas_marinas['periodo'], y=areas_marinas['valor'],
    mode='lines',
    line=dict(color=C_ODS14, width=3),
    fill='tozeroy', fillcolor='rgba(10,151,217,0.10)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Areas protegidas: <b>%{y:.2f}%</b><extra></extra>'
), row=1, col=1)

# Línea objetivo ODS (10%)
fig3.add_hline(
    y=10, line_width=1.5, line_dash='dash', line_color=C_DATO,
    annotation_text='Meta ODS: 10%',
    annotation_position='right',
    annotation_font=dict(size=9, color=C_DATO),
    row=1, col=1
)
fig3.add_annotation(
    x=2017, y=22.05,
    text='<b>2017: +20 pp</b><br>Decreto Revillagigedo<br>Archipielago (UNESCO)',
    showarrow=True, arrowhead=2, ax=-80, ay=-30,
    font=dict(size=9, color=C_ODS14),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_ODS14, borderwidth=1.5,
    row=1, col=1
)
fig3.add_annotation(
    x=2024, y=22.475,
    text='<b>22.5%</b><br>2024',
    showarrow=False, xanchor='left',
    font=dict(size=10, color=C_DATO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_DATO, borderwidth=1,
    row=1, col=1
)

# ── Panel 2: Manglares ────────────────────────────────────────────────────────
mang_vals = manglares['valor'] / 1000  # miles de ha
mang_colors = [C_VACIO if v < 790 else C_ESCASO if v < 850 else C_DATO for v in manglares['valor']]
fig3.add_trace(go.Bar(
    x=manglares['periodo'], y=mang_vals,
    marker_color=mang_colors,
    marker_line_color='white', marker_line_width=0.5,
    text=['{:.0f}k'.format(v) for v in mang_vals],
    textposition='outside', textfont=dict(size=9),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Manglares: <b>%{y:.0f} mil ha</b><extra></extra>'
), row=1, col=2)

fig3.add_annotation(
    x=2010, y=788,
    text='<b>Minimo 2010:</b><br>764k ha<br>-11% vs 1970',
    showarrow=True, arrowhead=2, ax=0, ay=-40,
    font=dict(size=9, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_VACIO, borderwidth=1,
    row=1, col=2
)
fig3.add_annotation(
    x=2020, y=935,
    text='<b>2020: 905k ha</b><br>Recuperacion +18%<br>desde el minimo',
    showarrow=True, arrowhead=2, ax=0, ay=-40,
    font=dict(size=9, color=C_DATO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_DATO, borderwidth=1.5,
    row=1, col=2
)

# ── Panel 3: Presupuesto investigación marina ─────────────────────────────────
pres_colors = [C_VACIO if v < 0.35 else C_ESCASO if v < 0.7 else C_DATO for v in presupuesto['valor']]
fig3.add_trace(go.Scatter(
    x=presupuesto['periodo'], y=presupuesto['valor'],
    mode='lines+markers',
    line=dict(color=C_ODS14, width=3),
    marker=dict(size=9, color=pres_colors, line=dict(width=1.5, color='white')),
    fill='tozeroy', fillcolor='rgba(10,151,217,0.08)',
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Presupuesto marina: <b>%{y:.3f}%</b><extra></extra>'
), row=1, col=3)

fig3.add_annotation(
    x=2012, y=1.329,
    text='<b>Pico 2012:</b><br>1.33%',
    showarrow=True, arrowhead=2, ax=45, ay=-20,
    font=dict(size=9, color=C_DATO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_DATO, borderwidth=1,
    row=1, col=3
)
fig3.add_annotation(
    x=2015, y=0.236,
    text='<b>Colapso 2015:</b><br>0.24%<br>-82% en 3 anos',
    showarrow=True, arrowhead=2, ax=55, ay=30,
    font=dict(size=9, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.9)', bordercolor=C_VACIO, borderwidth=1.5,
    row=1, col=3
)
fig3.add_annotation(
    x=2024, y=0.288,
    text='<b>2024: 0.29%</b><br>sin recuperacion<br>desde 2015',
    showarrow=False, xanchor='left',
    font=dict(size=10, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_VACIO, borderwidth=1.5,
    row=1, col=3
)

fig3.update_yaxes(showgrid=True, gridcolor='#E9ECEF')
fig3.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año')
fig3.update_yaxes(title_text='% zona maritima nacional', row=1, col=1)
fig3.update_yaxes(title_text='miles de hectareas', row=1, col=2)
fig3.update_yaxes(title_text='% presupuesto federal I+D', row=1, col=3)

fig3.update_layout(
    title=dict(
        text='<b>ODS 14 — El Oceano con Solo 4 Indicadores</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Areas protegidas: exito politico; presupuesto de investigacion: colapso del 78% desde 2012</span>',
        x=0.5, xanchor='center'
    ),
    height=480, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    margin=dict(t=100, b=60)
)
fig3.show()

## Acto IV — La Salud de los Datos
De los **333 indicadores** del sistema ODS de México:
- **41** no tienen **ningún** dato registrado
- **63** tienen apenas **1–4 observaciones** (menos que una medición por lustro)
- **78** tienen **5–14 observaciones** (pueden describir tendencias muy gruesas)
- **151** tienen **15 o más** observaciones (analizables con cierta confianza)

Es decir: **el 31 % de los indicadores es prácticamente inútil** para medir progreso.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 4 — La Salud de los Datos: distribución por ODS
# ══════════════════════════════════════════════════════════════════════════════

# Estadísticas globales
n_total = len(meta)
n_zero  = (meta['n_observations'] == 0).sum()
n_pobre = ((meta['n_observations'] > 0) & (meta['n_observations'] < 5)).sum()
n_medio = ((meta['n_observations'] >= 5) & (meta['n_observations'] < 15)).sum()
n_rico  = (meta['n_observations'] >= 15).sum()

print('Total: {} | Sin datos: {} | Escasos(1-4): {} | Medios(5-14): {} | Ricos(15+): {}'.format(
    n_total, n_zero, n_pobre, n_medio, n_rico
))

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Distribucion Global de Observaciones (333 indicadores)',
        'Indicadores por Calidad de Datos y ODS'
    ],
    specs=[[{'type': 'pie'}, {'type': 'xy'}]],
    column_widths=[0.35, 0.65],
    horizontal_spacing=0.10
)

# ── Panel 1: Pie chart global ─────────────────────────────────────────────────
fig4.add_trace(go.Pie(
    labels=['Sin datos<br>(0 obs)', 'Escasos<br>(1-4 obs)', 'Medios<br>(5-14 obs)', 'Ricos<br>(15+ obs)'],
    values=[n_zero, n_pobre, n_medio, n_rico],
    hole=0.45,
    marker_colors=[C_VACIO, C_ESCASO, C_MEDIO, C_DATO],
    textinfo='label+percent',
    textfont=dict(size=11),
    hovertemplate='<b>%{label}</b><br>N: <b>%{value}</b> indicadores (%{percent})<extra></extra>'
), row=1, col=1)

# ── Panel 2: Stacked bar por ODS ─────────────────────────────────────────────
fig4.add_trace(go.Bar(
    name='Sin datos (0)',
    x=['ODS ' + str(o) for o in cov['ods_number']],
    y=cov['n_zero'],
    marker_color=C_VACIO,
    marker_line_color='white', marker_line_width=0.3,
    hovertemplate='<b>%{x}</b><br>Sin datos: <b>%{y}</b><extra></extra>'
), row=1, col=2)

fig4.add_trace(go.Bar(
    name='Escasos (1-4)',
    x=['ODS ' + str(o) for o in cov['ods_number']],
    y=cov['n_escasos'],
    marker_color=C_ESCASO,
    marker_line_color='white', marker_line_width=0.3,
    hovertemplate='<b>%{x}</b><br>Escasos: <b>%{y}</b><extra></extra>'
), row=1, col=2)

fig4.add_trace(go.Bar(
    name='Medios (5-14)',
    x=['ODS ' + str(o) for o in cov['ods_number']],
    y=cov['n_medios'],
    marker_color=C_MEDIO,
    marker_line_color='white', marker_line_width=0.3,
    hovertemplate='<b>%{x}</b><br>Medios: <b>%{y}</b><extra></extra>'
), row=1, col=2)

fig4.add_trace(go.Bar(
    name='Ricos (15+)',
    x=['ODS ' + str(o) for o in cov['ods_number']],
    y=cov['n_ricos'],
    marker_color=C_DATO,
    marker_line_color='white', marker_line_width=0.3,
    hovertemplate='<b>%{x}</b><br>Ricos (15+): <b>%{y}</b><extra></extra>'
), row=1, col=2)

# Anotaciones destacadas
fig4.add_annotation(
    x='ODS 1', y=20,
    text='13 indicadores<br>sin ningun dato',
    showarrow=True, arrowhead=2, ax=0, ay=30,
    font=dict(size=9, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_VACIO, borderwidth=1,
    row=1, col=2
)
fig4.add_annotation(
    x='ODS 5', y=32,
    text='ODS 5: 0 indicadores<br>con 15+ datos',
    showarrow=True, arrowhead=2, ax=40, ay=-20,
    font=dict(size=9, color=C_ESCASO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_ESCASO, borderwidth=1,
    row=1, col=2
)

fig4.update_xaxes(tickangle=-45, row=1, col=2)
fig4.update_yaxes(title_text='Numero de indicadores', showgrid=True, gridcolor='#E9ECEF', row=1, col=2)

fig4.update_layout(
    title=dict(
        text='<b>La Salud de los Datos ODS — Distribucion de la Calidad de Monitoreo</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             '41 indicadores sin ningun dato; 104 mas con menos de 5 observaciones — 43% sin valor analitico</span>',
        x=0.5, xanchor='center'
    ),
    barmode='stack',
    height=500, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.22, xanchor='center', x=0.7,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=100, b=90)
)
fig4.show()

Total: 333 | Sin datos: 41 | Escasos(1-4): 93 | Medios(5-14): 109 | Ricos(15+): 90


## Acto V — 18 Años Sin Moverse
La comparación internacional deja al descubierto la magnitud real del rezago.
El **stunting infantil en México (15.1 %)** triplica el promedio de la OCDE (~5 %)
y casi duplica el promedio latinoamericano. A este ritmo de mejora —
0.3 puntos porcentuales en 18 años — **México necesitaría ~60 años más** para alcanzar
la meta ODS de <5 %. El problema no es sólo de datos: es de velocidad de acción.

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 5 — 18 Años Sin Moverse: Stunting en Perspectiva
# ══════════════════════════════════════════════════════════════════════════════

# ── Panel izquierdo: comparación internacional ────────────────────────────────
paises      = ['OCDE prom.', 'Chile', 'Brasil', 'Colombia', 'LAC prom.', 'Mexico', 'Bolivia']
stunting_cp = [4.6,          1.8,    4.5,      12.7,        11.0,        15.1,     16.1]
colores_cp  = [C_DATO, C_DATO, C_DATO, C_ESCASO, C_ESCASO, C_VACIO, C_VACIO]

# ── Panel derecho: México en el tiempo + proyección ──────────────────────────
stunting = get_serie('2.2.1')

# Proyección lineal
x_obs = np.array(stunting['periodo'])
y_obs = np.array(stunting['valor'])
m_pr, b_pr = np.polyfit(x_obs, y_obs, 1)
# Calcular año en que llega a 5%
ano_meta = int((5 - b_pr) / m_pr) if m_pr < 0 else 9999
x_proj = np.arange(2024, min(ano_meta + 5, 2090))

fig5 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Stunting Infantil — Comparacion Internacional (~2022)',
        'Mexico: Tendencia Historica y Proyeccion al Ritmo Actual'
    ],
    horizontal_spacing=0.12
)

# Panel izquierdo
fig5.add_trace(go.Bar(
    x=paises, y=stunting_cp,
    marker_color=colores_cp,
    marker_line_color='white', marker_line_width=0.5,
    text=['{:.1f}%'.format(v) for v in stunting_cp],
    textposition='outside', textfont=dict(size=10),
    showlegend=False,
    hovertemplate='<b>%{x}</b><br>Stunting: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=1)

fig5.add_hline(
    y=5, line_width=1.5, line_dash='dash', line_color=C_DATO,
    annotation_text='Meta ODS 2030: 5%',
    annotation_position='right',
    annotation_font=dict(size=9, color=C_DATO),
    row=1, col=1
)

# Panel derecho: serie real
fig5.add_trace(go.Scatter(
    x=stunting['periodo'], y=stunting['valor'],
    mode='lines+markers',
    name='Observado',
    line=dict(color=C_ODS2, width=3),
    marker=dict(size=10, color=C_ODS2, line=dict(width=1.5, color='white')),
    hovertemplate='<b>%{x}</b><br>Stunting: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

# Proyección
fig5.add_trace(go.Scatter(
    x=x_proj, y=np.clip(m_pr * x_proj + b_pr, 0, 20),
    mode='lines',
    name='Proyeccion al ritmo actual',
    line=dict(color=C_VACIO, width=2, dash='dot'),
    hovertemplate='<b>%{x}</b><br>Proyeccion: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

fig5.add_hline(
    y=5, line_width=1.5, line_dash='dash', line_color=C_DATO,
    annotation_text='Meta ODS <5%',
    annotation_position='right',
    annotation_font=dict(size=9, color=C_DATO),
    row=1, col=2
)

if ano_meta < 2090:
    fig5.add_annotation(
        x=ano_meta, y=5.5,
        text='<b>Meta alcanzada en ~{}</b><br>al ritmo actual'.format(ano_meta),
        showarrow=True, arrowhead=2, ax=0, ay=-40,
        font=dict(size=10, color=C_VACIO),
        bgcolor='rgba(255,255,255,0.9)', bordercolor=C_VACIO, borderwidth=1.5,
        row=1, col=2
    )
else:
    fig5.add_annotation(
        x=2060, y=9,
        text='<b>Al ritmo actual:</b><br>la meta ODS no se<br>alcanzaria este siglo',
        showarrow=False,
        font=dict(size=11, color=C_VACIO),
        bgcolor='rgba(255,255,255,0.9)', bordercolor=C_VACIO, borderwidth=2,
        row=1, col=2
    )

fig5.update_yaxes(title_text='% menores de 5 anos con stunting', showgrid=True, gridcolor='#E9ECEF')
fig5.update_xaxes(showgrid=True, gridcolor='#E9ECEF')
fig5.update_xaxes(title_text='Año', row=1, col=2)

fig5.update_layout(
    title=dict(
        text='<b>18 Anos Sin Moverse — Stunting Infantil en Mexico y el Mundo</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             'Mexico triplica el promedio OCDE; al ritmo actual tardara decadas en alcanzar la meta</span>',
        x=0.5, xanchor='center'
    ),
    height=500, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.18, xanchor='center', x=0.7,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=100, b=80)
)
fig5.show()

print('Tasa de mejora stunting: {:.4f} pp/año'.format(m_pr))
print('Para llegar a 5%: aprox. año {}'.format(ano_meta))

Tasa de mejora stunting: -0.0251 pp/año
Para llegar a 5%: aprox. año 2385


## Acto VI — El Presupuesto Hundido y los Indicadores Fantasma
La paradoja del ODS 14 mexicano: **proteger sin estudiar**. México decretó la mayor reserva
marina de su historia (22.5 % del mar territorial) mientras el presupuesto para estudiar ese
mar **caía un 78 %**. Peor aún: los indicadores que más importan para la gestión del océano
— pesca sostenible, acidificación, zonas de oxígeno mínimo, contaminación plástica —
**no existen** en el sistema ODS de México.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 6 — El Presupuesto Hundido y los Indicadores Fantasma
# ══════════════════════════════════════════════════════════════════════════════
presupuesto  = get_serie('14.A.1')
areas_mar    = get_serie('14.5.1')

# Indicadores ODS 14 globales de la ONU vs México
ind_globales = [
    '14.1.1 Contaminacion costera',
    '14.2.1 Gestion ecosistemas marinos',
    '14.3.1 Acidificacion del oceano',
    '14.4.1 Bancos pesqueros sostenibles',
    '14.4.2 Pesca ilegal (IUU)',
    '14.5.1 Areas marinas protegidas',
    '14.6.1 Subsidios pesca daninos',
    '14.7.1 Pesca en PIB paises insulares',
    '14.A.1 Investigacion marina (% I+D)',
    '14.B.1 Pesca artesanal — derechos',
    '14.C.1 UNCLOS ratificacion'
]
tiene_datos  = [False, False, False, False, False, True, False, False, True, False, False]
colores_ind  = [C_DATO if t else C_VACIO for t in tiene_datos]
textos_ind   = ['DATOS' if t else 'SIN DATOS' for t in tiene_datos]

fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Cobertura de Indicadores ODS 14 en Mexico',
        'Areas Protegidas vs Presupuesto de Investigacion Marina'
    ],
    column_widths=[0.45, 0.55],
    horizontal_spacing=0.12
)

# ── Panel 1: Inventario de indicadores ODS 14 ─────────────────────────────────
fig6.add_trace(go.Bar(
    x=[1 if t else 0.3 for t in tiene_datos],
    y=ind_globales,
    orientation='h',
    marker_color=colores_ind,
    marker_line_color='white', marker_line_width=0.5,
    text=textos_ind,
    textposition='inside',
    textfont=dict(size=10, color='white'),
    showlegend=False,
    hovertemplate='<b>%{y}</b><br>Estado: <b>%{text}</b><extra></extra>'
), row=1, col=1)

fig6.add_annotation(
    x=0.5, y=-0.5,
    text='<b>9 de 11 indicadores ODS 14 sin datos en Mexico</b>',
    showarrow=False,
    font=dict(size=11, color=C_VACIO),
    xref='x', yref='y',
    row=1, col=1
)

# ── Panel 2: Áreas protegidas vs presupuesto ──────────────────────────────────
# Combinar en años comunes
am_filt = areas_mar[areas_mar['periodo'] >= 2007].copy()
merged6 = am_filt[['periodo','valor']].rename(columns={'valor':'areas'}).merge(
    presupuesto[['periodo','valor']].rename(columns={'valor':'presup'}),
    on='periodo'
)

# Áreas marinas (eje principal)
fig6.add_trace(go.Scatter(
    x=merged6['periodo'], y=merged6['areas'],
    mode='lines+markers', name='Areas protegidas (%)',
    line=dict(color=C_DATO, width=3),
    marker=dict(size=8),
    hovertemplate='<b>%{x}</b><br>Areas protegidas: <b>%{y:.1f}%</b><extra></extra>'
), row=1, col=2)

# Presupuesto escalado para comparación visual
pres_escalado = merged6['presup'] * 17  # escala para visualización comparativa
fig6.add_trace(go.Scatter(
    x=merged6['periodo'], y=pres_escalado,
    mode='lines+markers', name='Presupuesto marina (x17 para comparar)',
    line=dict(color=C_VACIO, width=3, dash='dash'),
    marker=dict(size=8, symbol='square'),
    hovertemplate='<b>%{x}</b><br>Presup. marina (escalado): <b>%{y:.2f}</b><extra></extra>'
), row=1, col=2)

fig6.add_annotation(
    x=2016, y=18,
    text='Areas protegidas<br>se disparan...',
    showarrow=True, arrowhead=2, ax=-60, ay=-30,
    font=dict(size=9, color=C_DATO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_DATO, borderwidth=1,
    row=1, col=2
)
fig6.add_annotation(
    x=2015, y=0.236 * 17,
    text='...mientras el presupuesto<br>para estudiarlas colapsa',
    showarrow=True, arrowhead=2, ax=70, ay=20,
    font=dict(size=9, color=C_VACIO),
    bgcolor='rgba(255,255,255,0.85)', bordercolor=C_VACIO, borderwidth=1.5,
    row=1, col=2
)

fig6.update_xaxes(showgrid=True, gridcolor='#E9ECEF', title_text='Año', row=1, col=2)
fig6.update_yaxes(showgrid=True, gridcolor='#E9ECEF', title_text='% zona maritima', row=1, col=2)
fig6.update_xaxes(showgrid=False, row=1, col=1)
fig6.update_yaxes(showgrid=False, row=1, col=1)

fig6.update_layout(
    title=dict(
        text='<b>El Presupuesto Hundido y los Indicadores Fantasma del ODS 14</b><br>'
             '<span style="font-size:13px;color:#666;font-weight:normal">'
             '9 de 11 indicadores globales ODS 14 sin datos; investigacion marina colapso 78% desde 2012</span>',
        x=0.5, xanchor='center'
    ),
    height=500, plot_bgcolor=C_FONDO, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.18, xanchor='center', x=0.75,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=100, b=80)
)
fig6.show()

## Acto VII — El Radar del Silencio
Dos dimensiones del monitoreo para los 17 ODS:
- **Cobertura** (anillo interior): qué porcentaje de indicadores tiene algún dato
- **Profundidad** (anillo exterior): qué porcentaje tiene ≥15 observaciones (analizables)

La paradoja del ODS 14 queda expuesta: cobertura perfecta pero con tan pocos indicadores
que su "profundidad" no representa al sistema real. Los ODS 1, 2 y 10
son los peores en cobertura. El ODS 5 tiene el peor desempeño en profundidad a pesar de
tener buena cobertura — todos sus indicadores están desactualizados.

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURA 7 — El Radar del Silencio (17 ODS)
# ══════════════════════════════════════════════════════════════════════════════
ods_radar_labels = [
    'ODS1\nPobreza', 'ODS2\nHambre', 'ODS3\nSalud', 'ODS4\nEducacion',
    'ODS5\nGenero', 'ODS6\nAgua', 'ODS7\nEnergia', 'ODS8\nTrabajo',
    'ODS9\nIndustria', 'ODS10\nDesigualdad', 'ODS11\nCiudades',
    'ODS12\nConsumo', 'ODS13\nClima', 'ODS14\nOceanos',
    'ODS15\nTierra', 'ODS16\nPaz', 'ODS17\nAlianzas'
]

cov_scores  = list(cov['pct_datos'])      # % con algun dato
prof_scores = list(cov['pct_ricos'])      # % con 15+ datos

# Cerrar el radar (primer valor repetido al final)
theta  = ods_radar_labels + [ods_radar_labels[0]]
r_cov  = cov_scores  + [cov_scores[0]]
r_prof = prof_scores + [prof_scores[0]]

fig7 = go.Figure()

fig7.add_trace(go.Scatterpolar(
    r=r_cov,
    theta=theta,
    fill='toself',
    fillcolor='rgba(10,151,217,0.18)',
    line=dict(color=C_ODS14, width=2.5),
    name='Cobertura: % con algun dato',
    hovertemplate='<b>%{theta}</b><br>Cobertura: <b>%{r:.1f}%</b><extra></extra>'
))

fig7.add_trace(go.Scatterpolar(
    r=r_prof,
    theta=theta,
    fill='toself',
    fillcolor='rgba(45,198,83,0.20)',
    line=dict(color=C_DATO, width=2.5, dash='dash'),
    name='Profundidad: % con 15+ observaciones',
    hovertemplate='<b>%{theta}</b><br>Profundidad: <b>%{r:.1f}%</b><extra></extra>'
))

fig7.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 110],
            tickvals=[25, 50, 75, 100],
            ticktext=['25%', '50%', '75%', '100%'],
            gridcolor='#E9ECEF',
            linecolor='#DEE2E6'
        ),
        angularaxis=dict(
            tickfont=dict(size=9)
        )
    ),
    title=dict(
        text='<b>El Radar del Silencio — Cobertura y Profundidad de Monitoreo ODS (Mexico 2024)</b><br>'
             '<span style="font-size:12px;color:#666;font-weight:normal">'
             'Azul = % indicadores con algun dato; Verde punteado = % con datos suficientes para analisis</span>',
        x=0.5, xanchor='center'
    ),
    height=600, paper_bgcolor='white',
    font=dict(family='Inter, Arial, sans-serif', color=C_LINEA),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.08, xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.8)', bordercolor='#DEE2E6', borderwidth=1
    ),
    margin=dict(t=120, b=60)
)
fig7.show()

# Imprimir los peores
print('=== ODS con menor cobertura ===')
for _, row in cov.sort_values('pct_datos').head(5).iterrows():
    print('ODS {}: cobertura {:.1f}%, profundidad {:.1f}%'.format(
        int(row['ods_number']), row['pct_datos'], row['pct_ricos']))
print()
print('=== ODS con menor profundidad ===')
for _, row in cov.sort_values('pct_ricos').head(5).iterrows():
    print('ODS {}: cobertura {:.1f}%, profundidad {:.1f}%'.format(
        int(row['ods_number']), row['pct_datos'], row['pct_ricos']))

=== ODS con menor cobertura ===
ODS 1: cobertura 58.1%, profundidad 19.4%
ODS 2: cobertura 70.6%, profundidad 5.9%
ODS 4: cobertura 81.8%, profundidad 6.1%
ODS 10: cobertura 81.8%, profundidad 9.1%
ODS 3: cobertura 88.1%, profundidad 54.8%

=== ODS con menor profundidad ===
ODS 5: cobertura 93.1%, profundidad 0.0%
ODS 6: cobertura 100.0%, profundidad 4.0%
ODS 2: cobertura 70.6%, profundidad 5.9%
ODS 4: cobertura 81.8%, profundidad 6.1%
ODS 16: cobertura 88.9%, profundidad 8.3%


---
## Conclusiones: Lo Que No Se Mide No Se Gestiona

| ODS | Indicadores | Con datos | Con 15+ obs | Vacío crítico |
|-----|------------|-----------|-------------|---------------|
| **ODS 1** Pobreza | 31 | 18 (58%) | 6 (19%) | Protección social sin datos sub-nacionales |
| **ODS 2** Hambre | 17 | 12 (71%) | 1 (6%) | Stunting plano 18 años; precios alimentos sin medir |
| **ODS 5** Género | 29 | 27 (93%) | 0 (0%) | Cobertura engañosa: todos los datos son viejos |
| **ODS 10** Desigualdad | 11 | 9 (82%) | 1 (9%) | Gini desactualizado; distribución de riqueza invisible |
| **ODS 14** Océanos | 4 | 4 (100%) | 4 (100%) | Solo 4 indicadores: el 82% del marco global, ausente |

> **Mensaje clave**: México tiene un sistema de monitoreo ODS con enormes agujeros negros.
> El **ODS 1** no puede medir la pobreza a nivel municipal. El **ODS 2** lleva 18 años
> viendo el mismo número de niños con desnutrición. El **ODS 14** protege el 22 % del
> mar pero no tiene presupuesto para estudiarlo. El vacío estadístico no es un problema técnico:
> **es una decisión política sobre qué importa y qué se puede ignorar.**